
# Roxy notebook example: Pseudo amino acid composition (PseAAC) and variants

This notebook is a **reference implementation example** for the **PseAAC descriptor family** in Roxy.

PseAAC extends classical amino acid composition by incorporating **sequence-order information** through correlation factors computed from residue-level physicochemical properties.

## Covered outputs

This notebook implements:

- classical amino acid composition (AAC) baseline
- Type I PseAAC (composition + sequence-order correlation)
- configurable lambda (number of correlation tiers)
- configurable weight parameter `w`
- multi-property correlation using:
  - hydrophobicity
  - hydrophilicity
  - side-chain mass
- normalized descriptor vectors
- a simple variant using one-property PseAAC
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** that can later become part of the real Roxy package.


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "pseaac_1",
            "pseaac_2",
            "pseaac_3",
            "pseaac_4",
            "pseaac_5",
            "pseaac_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,pseaac_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,pseaac_2,GGGGGGGGGGGGGGG,B
2,pseaac_3,KRRKRRKRRKRRDDDDEE,A
3,pseaac_4,ACDEFGHIKLMNPQRSTVWY,B
4,pseaac_5,PPPPGSSSSSTTTTNNQQQ,A
5,pseaac_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and physicochemical properties

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

HYDROPHOBICITY = {
    "A": 0.62, "C": 0.29, "D": -0.90, "E": -0.74, "F": 1.19,
    "G": 0.48, "H": -0.40, "I": 1.38, "K": -1.50, "L": 1.06,
    "M": 0.64, "N": -0.78, "P": 0.12, "Q": -0.85, "R": -2.53,
    "S": -0.18, "T": -0.05, "V": 1.08, "W": 0.81, "Y": 0.26,
}

HYDROPHILICITY = {
    "A": -0.50, "C": -1.00, "D": 3.00, "E": 3.00, "F": -2.50,
    "G": 0.00, "H": -0.50, "I": -1.80, "K": 3.00, "L": -1.80,
    "M": -1.30, "N": 0.20, "P": 0.00, "Q": 0.20, "R": 3.00,
    "S": 0.30, "T": -0.40, "V": -1.50, "W": -3.40, "Y": -2.30,
}

SIDECHAIN_MASS = {
    "A": 15.0, "C": 47.0, "D": 59.0, "E": 73.0, "F": 91.0,
    "G": 1.0, "H": 82.0, "I": 57.0, "K": 72.0, "L": 57.0,
    "M": 75.0, "N": 58.0, "P": 41.0, "Q": 72.0, "R": 100.0,
    "S": 31.0, "T": 45.0, "V": 43.0, "W": 130.0, "Y": 107.0,
}

PSEAAC_PROPERTIES = {
    "hydrophobicity": HYDROPHOBICITY,
    "hydrophilicity": HYDROPHILICITY,
    "sidechain_mass": SIDECHAIN_MASS,
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def zscore_scale(scale: dict) -> dict:
    values = np.array([scale[aa] for aa in STANDARD_AA], dtype=float)
    mean = values.mean()
    std = values.std(ddof=0)
    return {aa: (scale[aa] - mean) / std for aa in STANDARD_AA}


def aac_frequencies(seq: str) -> dict:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return {aa: np.nan for aa in STANDARD_AA}
    return {aa: seq.count(aa) / len(seq) for aa in STANDARD_AA}


def correlation_theta(seq: str, lag: int, normalized_scales: dict) -> float:
    """Average squared difference across selected normalized physicochemical properties."""
    seq = clean_sequence(seq)
    n = len(seq)
    if n <= lag or lag < 1:
        return np.nan

    values = []
    for i in range(n - lag):
        aa1 = seq[i]
        aa2 = seq[i + lag]
        prop_diffs = []
        for _, scale in normalized_scales.items():
            prop_diffs.append((scale[aa1] - scale[aa2]) ** 2)
        values.append(np.mean(prop_diffs))

    return float(np.mean(values)) if len(values) > 0 else np.nan


def correlation_theta_single_property(seq: str, lag: int, normalized_scale: dict) -> float:
    seq = clean_sequence(seq)
    n = len(seq)
    if n <= lag or lag < 1:
        return np.nan

    values = []
    for i in range(n - lag):
        aa1 = seq[i]
        aa2 = seq[i + lag]
        values.append((normalized_scale[aa1] - normalized_scale[aa2]) ** 2)

    return float(np.mean(values)) if len(values) > 0 else np.nan


## Core Type I PseAAC implementation

In [5]:

def type1_pseaac(seq: str, lam: int = 5, w: float = 0.05, properties=None) -> dict:
    """Compute Type I PseAAC using multiple physicochemical properties."""
    seq = clean_sequence(seq)

    if properties is None:
        properties = ("hydrophobicity", "hydrophilicity", "sidechain_mass")

    out = {
        "pseaac_length": len(seq),
        "pseaac_valid_residue_count": len(seq),
        "pseaac_lambda": lam,
        "pseaac_weight": w,
    }

    if len(seq) == 0:
        return out

    aac = aac_frequencies(seq)

    normalized_scales = {
        prop: zscore_scale(PSEAAC_PROPERTIES[prop]) for prop in properties
    }

    thetas = []
    for lag in range(1, lam + 1):
        theta = correlation_theta(seq, lag, normalized_scales)
        thetas.append(theta if not np.isnan(theta) else 0.0)

    denominator = 1.0 + w * sum(thetas)

    for aa in STANDARD_AA:
        out[f"pseaac_{aa}"] = aac[aa] / denominator

    for i, theta in enumerate(thetas, start=1):
        out[f"pseaac_theta_{i}"] = (w * theta) / denominator

    out["pseaac_feature_sum"] = sum(
        out[f"pseaac_{aa}"] for aa in STANDARD_AA
    ) + sum(
        out[f"pseaac_theta_{i}"] for i in range(1, lam + 1)
    )

    return out


## One-property variant

In [6]:

def single_property_pseaac(seq: str, lam: int = 5, w: float = 0.05, property_name: str = "hydrophobicity") -> dict:
    """Simplified PseAAC variant using a single physicochemical property."""
    seq = clean_sequence(seq)

    out = {
        "spseaac_length": len(seq),
        "spseaac_valid_residue_count": len(seq),
        "spseaac_lambda": lam,
        "spseaac_weight": w,
        "spseaac_property": property_name,
    }

    if len(seq) == 0:
        return out

    aac = aac_frequencies(seq)
    normalized_scale = zscore_scale(PSEAAC_PROPERTIES[property_name])

    thetas = []
    for lag in range(1, lam + 1):
        theta = correlation_theta_single_property(seq, lag, normalized_scale)
        thetas.append(theta if not np.isnan(theta) else 0.0)

    denominator = 1.0 + w * sum(thetas)

    for aa in STANDARD_AA:
        out[f"spseaac_{aa}"] = aac[aa] / denominator

    for i, theta in enumerate(thetas, start=1):
        out[f"spseaac_theta_{i}"] = (w * theta) / denominator

    out["spseaac_feature_sum"] = sum(
        out[f"spseaac_{aa}"] for aa in STANDARD_AA
    ) + sum(
        out[f"spseaac_theta_{i}"] for i in range(1, lam + 1)
    )

    return out


## Functional usage on one sequence

In [7]:

example = type1_pseaac(df_demo.loc[0, "sequence"], lam=5, w=0.05)
list(example.items())[:18]


[('pseaac_length', 24),
 ('pseaac_valid_residue_count', 24),
 ('pseaac_lambda', 5),
 ('pseaac_weight', 0.05),
 ('pseaac_A', 0.027059184714329734),
 ('pseaac_C', 0.0),
 ('pseaac_D', 0.0),
 ('pseaac_E', 0.0),
 ('pseaac_F', 0.10823673885731894),
 ('pseaac_G', 0.027059184714329734),
 ('pseaac_H', 0.0),
 ('pseaac_I', 0.027059184714329734),
 ('pseaac_K', 0.027059184714329734),
 ('pseaac_L', 0.0811775541429892),
 ('pseaac_M', 0.027059184714329734),
 ('pseaac_N', 0.0),
 ('pseaac_P', 0.0),
 ('pseaac_Q', 0.0)]

## Apply Type I PseAAC to the full dataset

In [8]:

df_pseaac = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: type1_pseaac(x, lam=5, w=0.05)).apply(pd.Series),
    ],
    axis=1,
)

df_pseaac.head()


,sequence_id,sequence,label,pseaac_length,pseaac_valid_residue_count,pseaac_lambda,pseaac_weight,pseaac_A,pseaac_C,pseaac_D,...,pseaac_T,pseaac_V,pseaac_W,pseaac_Y,pseaac_theta_1,pseaac_theta_2,pseaac_theta_3,pseaac_theta_4,pseaac_theta_5,pseaac_feature_sum
0,pseaac_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,5.0,0.05,0.027059,0.000000,0.000000,...,0.027059,0.054118,0.027059,0.027059,0.072826,0.083458,0.081253,0.056084,0.056958,1.0
1,pseaac_2,GGGGGGGGGGGGGGG,B,15.0,15.0,5.0,0.05,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,pseaac_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,5.0,0.05,0.000000,0.000000,0.198179,...,0.000000,0.000000,0.000000,0.000000,0.016426,0.022004,0.010271,0.026479,0.033017,1.0
3,pseaac_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,5.0,0.05,0.032624,0.032624,0.032624,...,0.032624,0.032624,0.032624,0.032624,0.064500,0.071598,0.066783,0.083705,0.060944,1.0
4,pseaac_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,5.0,0.05,0.000000,0.000000,0.000000,...,0.201564,0.000000,0.000000,0.000000,0.004282,0.006107,0.008670,0.011338,0.012174,1.0


## Apply single-property variant

In [9]:

df_spseaac = pd.concat(
    [
        df_demo[["sequence_id", "sequence"]],
        df_demo["sequence"].apply(
            lambda x: single_property_pseaac(
                x,
                lam=5,
                w=0.05,
                property_name="hydrophobicity",
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_spseaac.head()


,sequence_id,sequence,spseaac_length,spseaac_valid_residue_count,spseaac_lambda,spseaac_weight,spseaac_property,spseaac_A,spseaac_C,spseaac_D,...,spseaac_T,spseaac_V,spseaac_W,spseaac_Y,spseaac_theta_1,spseaac_theta_2,spseaac_theta_3,spseaac_theta_4,spseaac_theta_5,spseaac_feature_sum
0,pseaac_1,MKWVTFISLLFLFSSAYSRGVFRR,24,24,5,0.05,hydrophobicity,0.026255,0.000000,0.000000,...,0.026255,0.052509,0.026255,0.026255,0.069956,0.100336,0.088450,0.056551,0.054593,1.0
1,pseaac_2,GGGGGGGGGGGGGGG,15,15,5,0.05,hydrophobicity,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,pseaac_3,KRRKRRKRRKRRDDDDEE,18,18,5,0.05,hydrophobicity,0.000000,0.000000,0.183157,...,0.000000,0.000000,0.000000,0.000000,0.025807,0.034696,0.016564,0.042427,0.056301,1.0
3,pseaac_4,ACDEFGHIKLMNPQRSTVWY,20,20,5,0.05,hydrophobicity,0.031124,0.031124,0.031124,...,0.031124,0.031124,0.031124,0.031124,0.066465,0.070543,0.063314,0.102857,0.074345,1.0
4,pseaac_5,PPPPGSSSSSTTTTNNQQQ,19,19,5,0.05,hydrophobicity,0.000000,0.000000,0.000000,...,0.201991,0.000000,0.000000,0.000000,0.003143,0.005244,0.007930,0.010958,0.013266,1.0


## Inspect PseAAC columns

In [10]:

pseaac_cols = [c for c in df_pseaac.columns if c.startswith("pseaac_") and c not in {"pseaac_length", "pseaac_valid_residue_count", "pseaac_lambda", "pseaac_weight"}]
len(pseaac_cols), pseaac_cols[:15]


(26,
 ['pseaac_A',
  'pseaac_C',
  'pseaac_D',
  'pseaac_E',
  'pseaac_F',
  'pseaac_G',
  'pseaac_H',
  'pseaac_I',
  'pseaac_K',
  'pseaac_L',
  'pseaac_M',
  'pseaac_N',
  'pseaac_P',
  'pseaac_Q',
  'pseaac_R'])

In [11]:

df_pseaac[
    [
        "sequence_id",
        "pseaac_A",
        "pseaac_C",
        "pseaac_D",
        "pseaac_theta_1",
        "pseaac_theta_2",
        "pseaac_theta_3",
        "pseaac_feature_sum",
    ]
]


,sequence_id,pseaac_A,pseaac_C,pseaac_D,pseaac_theta_1,pseaac_theta_2,pseaac_theta_3,pseaac_feature_sum
0,pseaac_1,0.027059,0.000000,0.000000,0.072826,0.083458,0.081253,1.0
1,pseaac_2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,pseaac_3,0.000000,0.000000,0.198179,0.016426,0.022004,0.010271,1.0
3,pseaac_4,0.032624,0.032624,0.032624,0.064500,0.071598,0.066783,1.0
4,pseaac_5,0.000000,0.000000,0.000000,0.004282,0.006107,0.008670,1.0
5,pseaac_6,0.000000,0.000000,0.032670,0.069552,0.061235,0.056337,1.0


## Dataset-level summary

In [12]:

pseaac_summary = (
    df_pseaac[pseaac_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

pseaac_summary.head(15)


,descriptor,mean_value
0,pseaac_feature_sum,1.000000
1,pseaac_G,0.190457
2,pseaac_R,0.090471
3,pseaac_S,0.070914
4,pseaac_K,0.059312
5,pseaac_T,0.054431
6,pseaac_P,0.049921
7,pseaac_D,0.043912
8,pseaac_theta_4,0.041076
9,pseaac_theta_2,0.040734


## Sanity checks

In [13]:

assert "pseaac_A" in df_pseaac.columns
assert "pseaac_theta_1" in df_pseaac.columns
assert "pseaac_theta_5" in df_pseaac.columns
assert "pseaac_feature_sum" in df_pseaac.columns
assert df_pseaac["pseaac_length"].min() > 0

assert np.allclose(df_pseaac["pseaac_feature_sum"], 1.0)
assert np.allclose(df_spseaac["spseaac_feature_sum"], 1.0)

print(f"Number of Type I PseAAC descriptor columns: {len(pseaac_cols)}")
print("PseAAC descriptor checks passed.")


Number of Type I PseAAC descriptor columns: 26
PseAAC descriptor checks passed.


## Class-style implementation closer to the real package

In [14]:

class PseAACDescriptors:
    """Example class-style PseAAC implementation for later migration into Roxy."""

    def __init__(self, lam: int = 5, w: float = 0.05, properties=None):
        self.lam = lam
        self.w = w
        self.properties = properties

    def transform_sequence(self, seq: str) -> dict:
        return type1_pseaac(
            seq,
            lam=self.lam,
            w=self.w,
            properties=self.properties,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


pseaac_transformer = PseAACDescriptors(lam=5, w=0.05)
pseaac_matrix = pseaac_transformer.transform(df_demo["sequence"].tolist())
pseaac_matrix.head()


,pseaac_length,pseaac_valid_residue_count,pseaac_lambda,pseaac_weight,pseaac_A,pseaac_C,pseaac_D,pseaac_E,pseaac_F,pseaac_G,...,pseaac_T,pseaac_V,pseaac_W,pseaac_Y,pseaac_theta_1,pseaac_theta_2,pseaac_theta_3,pseaac_theta_4,pseaac_theta_5,pseaac_feature_sum
0,24,24,5,0.05,0.027059,0.000000,0.000000,0.000000,0.108237,0.027059,...,0.027059,0.054118,0.027059,0.027059,0.072826,0.083458,0.081253,0.056084,0.056958,1.0
1,15,15,5,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,18,18,5,0.05,0.000000,0.000000,0.198179,0.099089,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.016426,0.022004,0.010271,0.026479,0.033017,1.0
3,20,20,5,0.05,0.032624,0.032624,0.032624,0.032624,0.032624,0.032624,...,0.032624,0.032624,0.032624,0.032624,0.064500,0.071598,0.066783,0.083705,0.060944,1.0
4,19,19,5,0.05,0.000000,0.000000,0.000000,0.000000,0.000000,0.050391,...,0.201564,0.000000,0.000000,0.000000,0.004282,0.006107,0.008670,0.011338,0.012174,1.0


## Merge transformer output back to the dataset

In [15]:

df_pseaac_class = pd.concat([df_demo, pseaac_matrix], axis=1)
df_pseaac_class.head()


,sequence_id,sequence,label,pseaac_length,pseaac_valid_residue_count,pseaac_lambda,pseaac_weight,pseaac_A,pseaac_C,pseaac_D,...,pseaac_T,pseaac_V,pseaac_W,pseaac_Y,pseaac_theta_1,pseaac_theta_2,pseaac_theta_3,pseaac_theta_4,pseaac_theta_5,pseaac_feature_sum
0,pseaac_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,5,0.05,0.027059,0.000000,0.000000,...,0.027059,0.054118,0.027059,0.027059,0.072826,0.083458,0.081253,0.056084,0.056958,1.0
1,pseaac_2,GGGGGGGGGGGGGGG,B,15,15,5,0.05,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,pseaac_3,KRRKRRKRRKRRDDDDEE,A,18,18,5,0.05,0.000000,0.000000,0.198179,...,0.000000,0.000000,0.000000,0.000000,0.016426,0.022004,0.010271,0.026479,0.033017,1.0
3,pseaac_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,5,0.05,0.032624,0.032624,0.032624,...,0.032624,0.032624,0.032624,0.032624,0.064500,0.071598,0.066783,0.083705,0.060944,1.0
4,pseaac_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,5,0.05,0.000000,0.000000,0.000000,...,0.201564,0.000000,0.000000,0.000000,0.004282,0.006107,0.008670,0.011338,0.012174,1.0



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move physicochemical property scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/pseaac.py`
- expose a class such as `PseAACDescriptors`
- allow configurable:
  - lambda
  - weight parameter
  - selected physicochemical properties
  - Type I vs simplified single-property variants
- add tests for:
  - empty sequences
  - very short sequences where lambda exceeds meaningful order depth
  - lower-case input
  - invalid characters removed during cleaning
  - vector normalization to 1.0


## Optional export

In [16]:
# df_pseaac.to_csv("demo_pseaac_descriptors.csv", index=False)
